Importing Libraries

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import norm

Loading Data

In [2]:
src = "participant_email_responses_view_rows.csv"
df = pd.read_csv(src)
df = df[~df['name'].isin(['Briant', 'B'])]

In [3]:
ground_truth = np.array([0, 1, 1, 1, 1, 1, 0, 0, 0, 0], dtype=bool)
phishing_cols = [f'is_phishing_email_{i}' for i in range(1, 11)]
rt_cols       = [f'response_time_email_{i}' for i in range(1, 11)]
conf_cols     = [f'confidence_level_email_{i}' for i in range(1, 11)]

group_map = {
    'control':  'Scenario 1: Control',
    'lime':     'Scenario 2: LIME Assisted',
    'word2vec': 'Scenario 3: IG Assisted',
}

n_phish = ground_truth.sum()     
n_legit = (~ground_truth).sum()  


Computing per-participant metrics

In [4]:
group_metrics = {}
for group in ['control', 'lime', 'word2vec']:
    g = df[df['group_type'] == group].dropna(subset=rt_cols + conf_cols)
    responses = g[phishing_cols].values.astype(bool)

    tp = np.sum((responses == True)  & (ground_truth == True),  axis=1)
    fp = np.sum((responses == True)  & (ground_truth == False), axis=1)
    tn = np.sum((responses == False) & (ground_truth == False), axis=1)

    hit_rate = (tp + 0.5) / (n_phish + 1)
    fa_rate  = (fp + 0.5) / (n_legit + 1)

    d_prime   = stats.norm.ppf(hit_rate) - stats.norm.ppf(fa_rate)
    c         = -0.5 * (stats.norm.ppf(hit_rate) + stats.norm.ppf(fa_rate))
    accuracy  = (tp + tn) / 10.0
    conf      = g[conf_cols].values / 5.0
    probs     = np.where(responses == True, conf, 1 - conf)
    brier     = np.mean((probs - ground_truth.astype(float)) ** 2, axis=1)
    latency_s = np.mean(g[rt_cols].values, axis=1) / 1000.0

    group_metrics[group] = {
        'accuracy': accuracy,
        'd_prime':  d_prime,
        'c':        c,
        'brier':    brier,
        'latency':  latency_s,
    }

Descriptive Statistics

In [8]:
metric_labels = {
    'accuracy': 'Accuracy',
    'd_prime':  "Sensitivity (d')",
    'c':        'Bias (c)',
    'brier':    'Brier Score',
    'latency':  'Latency (s)',
}

# Updated to match the row names in the provided table image
group_labels = {
    'control':  'Scenario 1: Control',
    'lime':     'Scenario 2: LIME Assisted',
    'word2vec': 'Scenario 3: IG Assisted',
}

print("=" * 125)
print(f"{'SECTION 2: DESCRIPTIVE STATISTICS (mean ± sd)':^125}")
print("=" * 125)

# 1. Print the header row
header_row = f"{'Scenario':<26}"
for mlabel in metric_labels.values():
    header_row += f" | {mlabel:<16}"
print(header_row)
print("-" * 125)

# 2. Iterate through groups (rows) first, then metrics (columns)
for g, glabel in group_labels.items():
    row_str = f"{glabel:<26}"
    for m in metric_labels.keys():
        vals = group_metrics[g][m]
        # Format the cell as 'mean ± sd'
        cell_val = f"{vals.mean():.4f} ± {vals.std():.4f}"
        row_str += f" | {cell_val:<16}"
    print(row_str)
    
print("-" * 125)

                                        SECTION 2: DESCRIPTIVE STATISTICS (mean ± sd)                                        
Scenario                   | Accuracy         | Sensitivity (d') | Bias (c)         | Brier Score      | Latency (s)     
-----------------------------------------------------------------------------------------------------------------------------
Scenario 1: Control        | 0.6423 ± 0.1780  | 0.6858 ± 0.9235  | 0.0663 ± 0.4179  | 0.2571 ± 0.0893  | 17.2221 ± 8.1938
Scenario 2: LIME Assisted  | 0.7950 ± 0.1284  | 1.5257 ± 0.7361  | -0.1621 ± 0.4011 | 0.1748 ± 0.1182  | 13.9675 ± 10.4833
Scenario 3: IG Assisted    | 0.7926 ± 0.1331  | 1.4741 ± 0.7257  | -0.0730 ± 0.4069 | 0.1645 ± 0.0812  | 9.6591 ± 6.6732 
-----------------------------------------------------------------------------------------------------------------------------


**Key Takeaways**
1. **Better Accuracy & Detection**: LIME & IG assitance overall boosted the accuracy from 64% to nearly 80%. XAi assistance also double the Sensitivitty (d') score, meaning users were much better at distinguishing "signal" from "noise" when assisted.

2. **Fasted Decisions Overall** (IG being the most efficient tool): XAi assistance helped users make decisions faster, indicated by Latency dropping from 17.2 seconds in Control Group, down to 13.9 with LIME, and further to 9.6 in IG.

3. **Higher Quality Predictions**: Using the Brier Score (lower is better), we see that assisted scenarios lowers the score from 0.25 in control to about 0.16 - 0.17, meaning that user's confidence and accuracy were better when they had help.

4. **Slight Shift in Bias**
    * **Control Group** (Bias = 0.06) means without any assistance, users were playing it safe when it comes to answering. They needed stronger, more obvious evidence before they were willing to make a positive identification
    * **Assisted Groups** (Bias = -0.07 to -0.16) means when the user were given LIME or IG assistance, their bias shifted into the negative. Essentially, AI tools lowered their mental thershold, meaning users felt more confident and required less of ther own internal evidence to say, "Yes, there's a signal here."


One-Way Anova

In [11]:
print("\n" + "=" * 65)
print("SECTION 3: ONE-WAY ANOVA")
print("=" * 65)

anova_results = {}
for m, mlabel in metric_labels.items():
    ctrl = group_metrics['control'][m]
    lime = group_metrics['lime'][m]
    ig   = group_metrics['word2vec'][m]

    _, p_val = stats.f_oneway(ctrl, lime, ig)
    anova_results[m] = {'p': p_val}
    sig = '* significant' if p_val < 0.05 else 'not significant'
    print(f"  {mlabel:<22} p = {p_val:.4f}  ({sig})")


SECTION 3: ONE-WAY ANOVA
  Accuracy               p = 0.0001  (* significant)
  Sensitivity (d')       p = 0.0001  (* significant)
  Bias (c)               p = 0.1660  (not significant)
  Brier Score            p = 0.0003  (* significant)
  Latency (s)            p = 0.0006  (* significant)


**Key Takeaways**
* Goal = "Is there a statistical significant difference somewhere within the three groups?"
* Rule = if p < 0.05, there is a significant difference somewhere, proceed to Post-HOC
* Results = Accuracy, Sensitivity, Brier, and Latency had p-values well below 0.05, meaning we can continue to Post-HOC Analysis
* Exepction = for Bias, ANOVA tells there is no stastically significant difference between any of the three groups, so we don't continue to Post-HOC

Post-HOC Pairwise t-Tests

In [7]:
bonferroni_alpha = 0.05 / 3  # 3 pairwise comparisons

print("\n" + "=" * 65)
print(f"SECTION 4: POST-HOC PAIRWISE t-TESTS (Bonferroni α = {bonferroni_alpha:.4f})")
print("=" * 65)

pairs = [
    ('control',  'lime',     'Control vs LIME'),
    ('control',  'word2vec', 'Control vs IG'),
    ('lime',     'word2vec', 'LIME vs IG'),
]

for m, mlabel in metric_labels.items():
    print(f"\n  {mlabel}:")

    if anova_results[m]['p'] >= 0.05:
        print(f"    Skipped — ANOVA not significant (p = {anova_results[m]['p']:.4f})")
        continue

    for g1, g2, pair_label in pairs:
        a = group_metrics[g1][m]
        b = group_metrics[g2][m]
        t_stat, p_val = stats.ttest_ind(a, b)
        sig = '* significant' if p_val < bonferroni_alpha else 'not significant'
        print(f"    {pair_label:<25}  t = {t_stat:.4f},  p = {p_val:.4f}  ({sig})")


SECTION 4: POST-HOC PAIRWISE t-TESTS (Bonferroni α = 0.0167)

  Accuracy:
    Control vs LIME            t = -3.1715,  p = 0.0028  (* significant)
    Control vs IG              t = -4.1671,  p = 0.0001  (* significant)
    LIME vs IG                 t = 0.0688,  p = 0.9453  (not significant)

  Sensitivity (d'):
    Control vs LIME            t = -3.2600,  p = 0.0022  (* significant)
    Control vs IG              t = -4.0998,  p = 0.0001  (* significant)
    LIME vs IG                 t = 0.2666,  p = 0.7906  (not significant)

  Bias (c):
    Skipped — ANOVA not significant (p = 0.1660)

  Brier Score:
    Control vs LIME            t = 2.6300,  p = 0.0117  (* significant)
    Control vs IG              t = 4.5613,  p = 0.0000  (* significant)
    LIME vs IG                 t = 0.4180,  p = 0.6772  (not significant)

  Latency (s):
    Control vs LIME            t = 1.1558,  p = 0.2540  (not significant)
    Control vs IG              t = 4.3435,  p = 0.0000  (* significant)
    LI

**Key Takeaways**
1. **Assistance works, but the type doesnt change accuracy**:
    * This can be seen by p values for LIME & IG consistently outperforming Control in Accuracy, Sensitivity and Brier Score. However when being matched up (LIME vs IG), the p-values are exceptionally high. Meaning that providing explanations drastically improves their detection performance, but neither LIME nor IG is statistically "better" at producing accurate decisions
2. **Integrated Gradients (IG) is the undisputed winner for speed**:
    * When looking at Latency, only Control vs IG that was highly significant (p= 0.0000). If the goal is to improve accuracy & efficiency, Integrated Gradients is the superior implementation